## 1  Distributional Counting

In [9]:
# 1.1
import sys
from collections import defaultdict

def count_cooccurrences(corpus_file, window_size, vocab, context_vocab):
    # Corpus file - WIKI-1 Percent dataset
    # Vocab is vocabvocab-15kws
    # context vocab is vocab-5k
    cooccurrence_counts = defaultdict(int)
    
    with open(corpus_file, 'r', encoding='utf-8') as f:
        for line in f:
            tokens = line.strip().split()
            for i, center_word in enumerate(tokens):
                if center_word not in vocab:
                    continue
                
                start = max(0, i - window_size)
                end = min(len(tokens), i + window_size + 1)
                
                for j in range(start, end):
                    if j != i:
                        context_word = tokens[j]
                        if context_word in context_vocab:
                            cooccurrence_counts[(center_word, context_word)] += 1
    
    return cooccurrence_counts

def load_vocab(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return set(line.strip() for line in f)

corpus_file = 'wiki-1percent.txt'
window_size = 3 
VC = 'vocab-5k.txt'
V = 'vocab-15kws.txt'

vocab = load_vocab(V)
context_vocab = load_vocab(VC)

cooccurrence_counts = count_cooccurrences(corpus_file, window_size, vocab, context_vocab)


# Print only the first 100 co-occurrences
for (x, y), count in list(cooccurrence_counts.items())[:100]:
    print(f"<{x}, {y}>: {count}")



<they, encouraged>: 7
<they, playing>: 49
<they, with>: 801
<encouraged, they>: 7
<encouraged, playing>: 1
<encouraged, with>: 5
<encouraged, sexual>: 3
<playing, they>: 49
<playing, encouraged>: 1
<playing, with>: 296
<playing, sexual>: 1
<playing, roles>: 20
<with, they>: 801
<with, encouraged>: 5
<with, playing>: 296
<with, sexual>: 78
<with, roles>: 47
<with, and>: 16238
<sexual, encouraged>: 3
<sexual, playing>: 1
<sexual, with>: 78
<sexual, roles>: 1
<sexual, and>: 188
<roles, playing>: 20
<roles, with>: 47
<roles, sexual>: 1
<roles, and>: 209
<roles, ,>: 209
<and, with>: 16238
<and, sexual>: 188
<and, roles>: 209
<and, ,>: 202815
<and, in>: 58245
<sexuality, sexual>: 5
<sexuality, roles>: 1
<sexuality, and>: 65
<sexuality, ,>: 60
<sexuality, in>: 23
<sexuality, europe>: 1
<,, roles>: 209
<,, and>: 202815
<,, in>: 143455
<,, europe>: 1161
<,, the>: 271463
<in, and>: 58245
<in, ,>: 143455
<in, europe>: 1526
<in, the>: 191850
<in, main>: 765
<europe, ,>: 1161
<europe, in>: 1526
<eu

In [10]:
#1.2 

corpus_file = 'wiki-1percent.txt'
vocab_file = 'vocab-15kws.txt'  
context_vocab_file = 'vocab-5k.txt'  

V = load_vocab(vocab_file)
VC = load_vocab(context_vocab_file)


cooccurrence_counts_3 = count_cooccurrences(corpus_file, 3, V, VC)


cooccurrence_counts_6 = count_cooccurrences(corpus_file, 6, V, VC)


def print_counts(pairs, counts_3, counts_6):
    for x, y in pairs:
        count_3 = counts_3.get((x, y), 0)
        count_6 = counts_6.get((x, y), 0)
        print(f" <{x}, {y}>: #(x,y) for w=3: {count_3}, for w=6: {count_6}")


pairs_to_check = [
    ('chicken', 'the'),
    ('chicken', 'wings'),
    ('chicago', 'chicago'),
    ('coffee', 'the'),
    ('coffee', 'cup'),
    ('coffee', 'coffee')
]

print_counts(pairs_to_check, cooccurrence_counts_3, cooccurrence_counts_6)




 <chicken, the>: #(x,y) for w=3: 52, for w=6: 103
 <chicken, wings>: #(x,y) for w=3: 6, for w=6: 7
 <chicago, chicago>: #(x,y) for w=3: 38, for w=6: 122
 <coffee, the>: #(x,y) for w=3: 95, for w=6: 201
 <coffee, cup>: #(x,y) for w=3: 10, for w=6: 14
 <coffee, coffee>: #(x,y) for w=3: 4, for w=6: 36


In [11]:
#1.3 evaluate your count-based word vectors using EVALWS and report your results on MEN and SimLex-999
import numpy as np
from scipy.stats import spearmanr


def create_word_vectors(cooccurrence_counts, vocab, context_vocab):
    word_vectors = {}
    for word in vocab:
        vector = [cooccurrence_counts.get((word, context_word), 0) for context_word in context_vocab]
        word_vectors[word] = np.array(vector)
    return word_vectors

def cosine_similarity(v1, v2):
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(v1, v2) / (norm1 * norm2)

def load_similarity_dataset(file_path):
    pairs = []
    scores = []
    with open(file_path, 'r', encoding='utf-8') as f:
        next(f) # SKIP String Values
        for line in f:
            word1, word2, score = line.strip().split('\t')
            pairs.append((word1, word2))
            scores.append(float(score))
    return pairs, scores

def evaluate_word_vectors(word_vectors, similarity_file):
    pairs, human_scores = load_similarity_dataset(similarity_file)
    cosine_scores = []
    
    for word1, word2 in pairs:
        if word1 in word_vectors and word2 in word_vectors:
            sim = cosine_similarity(word_vectors[word1], word_vectors[word2])
        else:
            sim = 0.0
        cosine_scores.append(sim)
    
    correlation, _ = spearmanr(human_scores, cosine_scores)
    return correlation


corpus_file = 'wiki-1percent.txt'
vocab_file = 'vocab-15kws.txt'  # V
context_vocab_file = 'vocab-5k.txt'  # VC
window_size = 3

V = load_vocab(vocab_file)
VC = load_vocab(context_vocab_file)

cooccurrence_counts = count_cooccurrences(corpus_file, window_size, V, VC)
word_vectors = create_word_vectors(cooccurrence_counts, V, VC)


men_correlation = evaluate_word_vectors(word_vectors, 'men.txt')
simlex_correlation = evaluate_word_vectors(word_vectors, 'simlex-999.txt')

print(f"Spearman's ρ for MEN: {men_correlation}")
print(f"Spearman's ρ for SimLex-999: {simlex_correlation}")





Spearman's ρ for MEN: 0.2251396048448754
Spearman's ρ for SimLex-999: 0.05876135331349779


## 2 Combining Counts with Inverse Document Frequency (IDF)

In [15]:

import math
from collections import defaultdict, Counter

def compute_idf_counts(corpus_file, V, VC, w):
    word_counts = defaultdict(Counter)
    sentence_counts = Counter()
    total_sentences = 0

    with open(corpus_file, 'r', encoding='utf-8') as f:
        for line in f:
            words = line.strip().lower().split()
            total_sentences += 1
            sentence_words = set()
            
            for i, word in enumerate(words):
                if word in V:
                    start = max(0, i - w)
                    end = min(len(words), i + w + 1)
                    for j in range(start, end):
                        if i != j and words[j] in VC:
                            word_counts[word][words[j]] += 1
                            sentence_words.add(words[j])
            
            for context_word in sentence_words:
                sentence_counts[context_word] += 1

    return word_counts, sentence_counts, total_sentences

def compute_idf_word_vectors(word_counts, sentence_counts, total_sentences, V, VC):
    word_vectors = {}

    for word in V:
        vector = []
        for context_word in VC:
            tf = word_counts[word][context_word]
            idf = math.log(total_sentences / (sentence_counts[context_word] + 1))
            vector.append(tf * idf)
        word_vectors[word] = np.array(vector)

    return word_vectors


V = load_vocab('vocab-15kws.txt')
VC = load_vocab('vocab-5k.txt')


w = 3
corpus_file = 'wiki-1percent.txt'
word_counts, sentence_counts, total_sentences = compute_idf_counts(corpus_file, V, VC, w)
idf_word_vectors = compute_idf_word_vectors(word_counts, sentence_counts, total_sentences, V, VC)


men_correlation = evaluate_word_vectors(idf_word_vectors, 'men.txt')
simlex_correlation = evaluate_word_vectors(idf_word_vectors, 'simlex-999.txt')

print(f"IDF-based Spearman's ρ for MEN: {men_correlation}")
print(f"IDF-based Spearman's ρ for SimLex-999: {simlex_correlation}")


IDF-based Spearman's ρ for MEN: 0.24937762833299582
IDF-based Spearman's ρ for SimLex-999: 0.07289683751770486


## 3 Pointwise Mutual Information (PMI)

In [17]:
#3.1 For center word x = “coffee”, print the 10 context words with the largest PMIs and the 10 context words with the smallest PMIs.
from collections import defaultdict
from math import log2

def read_vocabulary(file_path):
    """Read a vocabulary file and return a set of words."""
    vocabulary_set = set()
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            word = line.strip()  # Remove whitespace
            vocabulary_set.add(word)
    return vocabulary_set

def compute_pmi(counts, vocab, context_vocab):
    """Calculate the Pointwise Mutual Information (PMI) for word pairs."""
    N = sum(counts.values())  # Total co-occurrences
    
    # Count occurrences for marginal probabilities
    x_counts = defaultdict(int)
    y_counts = defaultdict(int)
    for (x, y), count in counts.items():
        x_counts[x] += count
        y_counts[y] += count
    
    pmi = {}
    for x in vocab:
        for y in context_vocab:
            if (x, y) in counts:
                pmi[(x, y)] = log2((counts[(x, y)] * N) / (x_counts[x] * y_counts[y]))
            else:
                pmi[(x, y)] = 0  # PMI is zero if there are no counts
    
    return pmi

def print_top_pmi_for_word(pmi, word, context_vocab, top_n=10):
    """Print the top and bottom PMI values for a given word."""
    word_pmi = [(context_word, pmi[(word, context_word)]) for context_word in context_vocab if (word, context_word) in pmi]
    word_pmi.sort(key=lambda x: x[1], reverse=True)
    
    print(f"Top {top_n} context words with highest PMI for '{word}':")
    for context_word, pmi_value in word_pmi[:top_n]:
        print(f"{context_word}: {pmi_value:.4f}")
    
    print(f"\nBottom {top_n} context words with lowest PMI for '{word}':")
    for context_word, pmi_value in word_pmi[-top_n:]:
        print(f"{context_word}: {pmi_value:.4f}")


corpus_path = "wiki-1percent.txt"
Vocab = "vocab-15kws.txt"
ContextVocab = "vocab-5k.txt"
window_size = 3


V = read_vocabulary(Vocab)
VC = read_vocabulary(ContextVocab)


counts = count_cooccurrences(corpus_path, window_size, V, VC)
pmi = compute_pmi(counts, V, VC)


print_top_pmi_for_word(pmi, "coffee", VC)


Top 10 context words with highest PMI for 'coffee':
tea: 8.1660
drinking: 7.5880
shop: 7.4117
costa: 7.3503
shops: 7.2608
sugar: 6.5339
coffee: 6.5020
mix: 6.1312
seattle: 5.9508
houses: 5.8682

Bottom 10 context words with lowest PMI for 'coffee':
page: -1.2806
when: -1.4043
more: -1.4785
after: -1.5985
its: -1.8395
not: -1.9116
this: -1.9795
had: -1.9875
be: -2.1510
he: -2.2603


In [17]:
1.2 # Evaluate (EVALWS) your PMI-based word vectors and report your results.

import numpy as np
from scipy.stats import spearmanr

def compute_pmi_word_vectors(word_counts, sentence_counts, total_sentences, V, VC):
    word_vectors = {}
    for word in V:
        if word in word_counts:
            vector = []
            for context_word in VC:
                if context_word in word_counts[word]:
                    
                    pmi = np.log2(((word_counts[word][context_word] + 1) * (total_sentences + 1)) / 
                                  ((sentence_counts[word] + 1) * (sentence_counts[context_word] + 1)))
                    vector.append(max(pmi, 0))  
                else:
                    vector.append(0)
            word_vectors[word] = np.array(vector)
    return word_vectors

def cosine_similarity(v1, v2):
    
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(v1, v2) / (norm1 * norm2)

def evaluate_word_vectors(word_vectors, similarity_file):
    """Evaluate word vectors using a similarity dataset."""
    pairs, human_scores = load_similarity_dataset(similarity_file)
    cosine_scores = []
    
    for word1, word2 in pairs:
        if word1 in word_vectors and word2 in word_vectors:
            sim = cosine_similarity(word_vectors[word1], word_vectors[word2])
        else:
            sim = 0.0
        cosine_scores.append(sim)
    
    correlation = spearmanr(human_scores, cosine_scores)[0]
    return correlation


V = load_vocab('vocab-15kws.txt')
VC = load_vocab('vocab-5k.txt')


w = 3
corpus_file = 'wiki-1percent.txt'
word_counts, sentence_counts, total_sentences = compute_idf_counts(corpus_file, V, VC, w)
pmi_word_vectors = compute_pmi_word_vectors(word_counts, sentence_counts, total_sentences, V, VC)


men_correlation = evaluate_word_vectors(pmi_word_vectors, 'men.txt')
simlex_correlation = evaluate_word_vectors(pmi_word_vectors, 'simlex-999.txt')

print(f"PMI-based Spearman's for MEN: {men_correlation:.4f}")
print(f"PMI-based Spearman's for SimLex-999: {simlex_correlation:.4f}")




PMI-based Spearman's for MEN: 0.4626
PMI-based Spearman's for SimLex-999: 0.2159


# #5 Qualitative Analysis

In [25]:
# 5.1  For the two window sizes w = 1 and w = 6, compute and print the 10 nearest neighbors for the query word judges.
def create_pmi_vectors_for_window(corpus_path, V, VC, window_size):
    
    counts = count_cooccurrences(corpus_path, window_size, V, VC)
    pmi = compute_pmi(counts, V, VC)
    return {x: {y: pmi.get((x, y), 0) for y in VC} for x in V}

In [27]:
def cosine_similarity(v1, v2):
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(v1, v2) / (norm1 * norm2)

In [29]:
def find_nearest_neighbors(pmi_vectors, query_word, k=10):
    
    if query_word not in pmi_vectors:
        return []

    query_vector = np.array([pmi_vectors[query_word].get(context_word, 0) for context_word in pmi_vectors[query_word]])
    
    similarities = {}
    for word in pmi_vectors:
        if word != query_word:  # Skip the query word
            neighbor_vector = np.array([pmi_vectors[word].get(context_word, 0) for context_word in pmi_vectors[word]])
            sim = cosine_similarity(query_vector, neighbor_vector)
            similarities[word] = sim
    
    # Sort neighbors by similarity and return the top 10
    nearest_neighbors = sorted(similarities.items(), key=lambda item: item[1], reverse=True)
    return nearest_neighbors[:k]

In [31]:
# Main execution block
corpus_path = "wiki-1percent.txt"
Vocab = "vocab-15kws.txt"
contextVocab = "vocab-5k.txt"


V = read_vocabulary(Vocab)
VC = read_vocabulary(contextVocab)


pmi_vectors_w1 = create_pmi_vectors_for_window(corpus_path, V, VC, window_size=1)

nearest_neighbors_w1 = find_nearest_neighbors(pmi_vectors_w1, "judges", k=10)

print("Nearest neighbors for 'judges' with window size w = 1:")
for neighbor, similarity in nearest_neighbors_w1:
    print(f"{neighbor}: {similarity:.4f}")



Nearest neighbors for 'judges' with window size w = 1:
judge: 0.2225
players: 0.2115
appeals: 0.1893
officials: 0.1817
ministers: 0.1786
justices: 0.1784
leaders: 0.1729
members: 0.1729
unanimously: 0.1700
contestants: 0.1657


In [31]:
pmi_vectors_w6 = create_pmi_vectors_for_window(corpus_path, V, VC, window_size=6)

nearest_neighbors_w6 = find_nearest_neighbors(pmi_vectors_w6, "judges", k=10)

print("\nNearest neighbors for 'judges' with window size w = 6:")
for neighbor, similarity in nearest_neighbors_w6:
    print(f"{neighbor}: {similarity:.4f}")


Nearest neighbors for 'judges' with window size w = 6:
judge: 0.3054
jury: 0.2880
appeals: 0.2774
courts: 0.2740
panel: 0.2740
supreme: 0.2688
justice: 0.2566
contestants: 0.2565
candidates: 0.2488
appeal: 0.2488


In [33]:
#5.2 different part-of-speech tags for the query word
import numpy as np
from collections import defaultdict
from math import log2
import pandas as pd
from scipy.stats import spearmanr


def read_vocabulary(file_path):
    """Read a vocabulary file and return a set of words."""
    vocabulary_set = set()
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            word = line.strip()  # Remove whitespace
            vocabulary_set.add(word)
    return vocabulary_set


In [35]:
def categorize_words(words):
    pos_dict = {
        'nouns': ["coffee", "judges"],
        'verbs': ["transported"],
        'adjectives': ["large"],
        'prepositions': ["under"]
    }
    return {key: [word for word in words if word in value] for key, value in pos_dict.items()}


In [37]:
def analyze_nearest_neighbors(pmi_vectors, query_words, window_sizes):
    for window_size in window_sizes:
        pmi_vectors_current = create_pmi_vectors_for_window(corpus_path, V, VC, window_size)
        print(f"\nAnalyzing nearest neighbors with window size = {window_size}:\n")
        
        for word in query_words:
            nearest_neighbors = find_nearest_neighbors(pmi_vectors_current, word, k=10)
            print(f"Nearest neighbors for '{word}':")
            for neighbor, similarity in nearest_neighbors:
                print(f"  {neighbor}: {similarity:.4f}")
            print() 


In [42]:
corpus_path = "wiki-1percent.txt"
Vocab = "vocab-15kws.txt"
ContextVocab = "vocab-5k.txt"


V = read_vocabulary(Vocab)
VC = read_vocabulary(ContextVocab)


window_sizes = [1, 6]


query_words = [
    "coffee", "judges",  # Nouns
    "transported",    # Verbs
    "large",        # Adjectives
    "under"      # Prepositions
]


analyze_nearest_neighbors(V, query_words, window_sizes)


Analyzing nearest neighbors with window size = 1:

Nearest neighbors for 'coffee':
  tea: 0.2628
  donut: 0.2050
  chocolate: 0.1938
  candy: 0.1931
  wine: 0.1908
  jewelry: 0.1735
  beer: 0.1640
  shoe: 0.1603
  clothing: 0.1562
  food: 0.1554

Nearest neighbors for 'judges':
  judge: 0.2225
  players: 0.2115
  appeals: 0.1893
  officials: 0.1817
  ministers: 0.1786
  justices: 0.1784
  leaders: 0.1729
  members: 0.1729
  unanimously: 0.1700
  contestants: 0.1657

Nearest neighbors for 'transported':
  shipped: 0.2915
  transmitted: 0.2610
  relegated: 0.2438
  converted: 0.2408
  reassigned: 0.2309
  marched: 0.2282
  traced: 0.2259
  carried: 0.2175
  sent: 0.2167
  sailed: 0.2157

Nearest neighbors for 'large':
  small: 0.5499
  larger: 0.3312
  largest: 0.3304
  smaller: 0.3156
  major: 0.2979
  huge: 0.2958
  significant: 0.2942
  main: 0.2605
  great: 0.2602
  vast: 0.2576

Nearest neighbors for 'under':
  within: 0.3044
  during: 0.2832
  through: 0.2764
  into: 0.2625
  over

In [44]:
#5.3 words with multiple senses
window_sizes = [1,6]
query_words = ["bank",   "cell",   "apple",   "apples", "axes",   "frame",  "light",  "well"  ]

analyze_nearest_neighbors(V, query_words, window_sizes)


Analyzing nearest neighbors with window size = 1:

Nearest neighbors for 'bank':
  side: 0.2065
  coast: 0.2063
  railway: 0.2063
  park: 0.2022
  africa: 0.1993
  banks: 0.1965
  corporation: 0.1927
  property: 0.1870
  railroad: 0.1847
  province: 0.1836

Nearest neighbors for 'cell':
  cells: 0.3019
  tissue: 0.2034
  tissues: 0.1708
  human: 0.1693
  brain: 0.1676
  outer: 0.1585
  engine: 0.1558
  cellular: 0.1550
  model: 0.1549
  skin: 0.1526

Nearest neighbors for 'apple':
  pine: 0.1844
  atari: 0.1620
  cherry: 0.1529
  christmas: 0.1412
  olive: 0.1401
  bear: 0.1374
  mini: 0.1339
  desktop: 0.1314
  egg: 0.1307
  oak: 0.1304

Nearest neighbors for 'apples':
  tomatoes: 0.2025
  flowers: 0.1920
  impatient: 0.1766
  grapes: 0.1731
  guys: 0.1719
  shrubs: 0.1647
  classmates: 0.1645
  dreams: 0.1627
  algae: 0.1560
  arrows: 0.1538

Nearest neighbors for 'axes':
  tributaries: 0.1897
  branches: 0.1702
  phases: 0.1636
  viewpoints: 0.1589
  dimensions: 0.1563
  ridges: 0.

## 4 Quantitative Comparisons

In [52]:

window_sizes = [1, 3, 6]
context_vocabs = ['vocab-15kws.txt', 'vocab-5k.txt']
methods = ['counts', 'idf']
evaluation_datasets = ['men.txt', 'simlex-999.txt']

# Load main vocabulary
V = load_vocab('vocab-15kws.txt')

print("Evaluating word vectors for different configurations...")
results = {}

def compute_count_word_vectors(word_counts, V, VC):
    word_vectors = {}
    for word in V:
        if word in word_counts:
            vector = np.array([word_counts[word].get(context_word, 0) for context_word in VC])
            word_vectors[word] = vector
    return word_vectors

def compute_idf_word_vectors(word_counts, sentence_counts, total_sentences, V, VC):
    word_vectors = {}
    for word in V:
        if word in word_counts:
            vector = []
            for context_word in VC:
                tf = word_counts[word][context_word]
                idf = np.log(total_sentences / (sentence_counts[context_word] + 1))
                vector.append(tf * idf)
            word_vectors[word] = np.array(vector)
    return word_vectors

for method in methods:
    for w in window_sizes:
        for vc_file in context_vocabs:
            VC = load_vocab(vc_file)
            
            print(f"\nComputing {method.upper()} word vectors (w={w}, VC={vc_file})...")
            word_counts, sentence_counts, total_sentences = compute_idf_counts(corpus_file, V, VC, w)
            
            if method == 'counts':
                word_vectors = compute_count_word_vectors(word_counts, V, VC)
            elif method == 'idf':
                word_vectors = compute_idf_word_vectors(word_counts, sentence_counts, total_sentences, V, VC)
            
            for dataset in evaluation_datasets:
                correlation = evaluate_word_vectors(word_vectors, dataset)
                key = (method, w, vc_file, dataset)
                results[key] = correlation
                print(f"{dataset} correlation: {correlation:.4f}")


print("\nSummary of Results:")
print("Method\tWindow\tContext Vocab\tMEN\tSimLex-999")
for method in methods:
    for w in window_sizes:
        for vc_file in context_vocabs:
            men_corr = results[(method, w, vc_file, 'men.txt')]
            simlex_corr = results[(method, w, vc_file, 'simlex-999.txt')]
            print(f"{method}\t{w}\t{vc_file}\t{men_corr:.4f}\t{simlex_corr:.4f}")



print("1. Effect of window size:")
for method in methods:
    print(f"  - For {method.upper()} method:")
    for vc_file in context_vocabs:
        men_trends = [results[(method, w, vc_file, 'men.txt')] for w in window_sizes]
        simlex_trends = [results[(method, w, vc_file, 'simlex-999.txt')] for w in window_sizes]
        print(f"    - With {vc_file}: MEN trend = {men_trends}, SimLex trend = {simlex_trends}")




Evaluating word vectors for different configurations...

Computing COUNTS word vectors (w=1, VC=vocab-15kws.txt)...
men.txt correlation: 0.2064
simlex-999.txt correlation: 0.0700

Computing COUNTS word vectors (w=1, VC=vocab-5k.txt)...
men.txt correlation: 0.2091
simlex-999.txt correlation: 0.0678

Computing COUNTS word vectors (w=3, VC=vocab-15kws.txt)...
men.txt correlation: 0.2208
simlex-999.txt correlation: 0.0571

Computing COUNTS word vectors (w=3, VC=vocab-5k.txt)...
men.txt correlation: 0.2251
simlex-999.txt correlation: 0.0588

Computing COUNTS word vectors (w=6, VC=vocab-15kws.txt)...
men.txt correlation: 0.2369
simlex-999.txt correlation: 0.0407

Computing COUNTS word vectors (w=6, VC=vocab-5k.txt)...
men.txt correlation: 0.2411
simlex-999.txt correlation: 0.0447

Computing IDF word vectors (w=1, VC=vocab-15kws.txt)...
men.txt correlation: 0.2405
simlex-999.txt correlation: 0.1327

Computing IDF word vectors (w=1, VC=vocab-5k.txt)...
men.txt correlation: 0.2374
simlex-999.tx

In [50]:

window_sizes = [1, 3, 6]
context_vocabs = ['vocab-15kws.txt', 'vocab-5k.txt']
methods = ['pmi']  
evaluation_datasets = ['men.txt', 'simlex-999.txt']


V = load_vocab('vocab-15kws.txt')

print("Evaluating word vectors for different configurations...")
results = {}

def compute_pmi_word_vectors(word_counts, sentence_counts, total_sentences, V, VC):
    word_vectors = {}
    for word in V:
        if word in word_counts:
            vector = []
            for context_word in VC:
                if context_word in word_counts[word]:
                    
                    pmi = np.log2(((word_counts[word][context_word] + 1) * (total_sentences + 1)) / 
                                  ((sentence_counts[word] + 1) * (sentence_counts[context_word] + 1)))
                    vector.append(max(pmi, 0))  # Using positive PMI
                else:
                    vector.append(0)
            word_vectors[word] = np.array(vector)
    return word_vectors

for w in window_sizes:
    for vc_file in context_vocabs:
        VC = load_vocab(vc_file)
        
        print(f"\nComputing PMI word vectors (w={w}, VC={vc_file})...")
        word_counts, sentence_counts, total_sentences = compute_idf_counts(corpus_file, V, VC, w)
        
        word_vectors = compute_pmi_word_vectors(word_counts, sentence_counts, total_sentences, V, VC)
        
        for dataset in evaluation_datasets:
            correlation = evaluate_word_vectors(word_vectors, dataset)
            key = ('pmi', w, vc_file, dataset)
            results[key] = correlation
            print(f"{dataset} correlation: {correlation:.4f}")


print("\nSummary of Results:")
print("Method\tWindow\tContext Vocab\tMEN\tSimLex-999")
for w in window_sizes:
    for vc_file in context_vocabs:
        men_corr = results[('pmi', w, vc_file, 'men.txt')]
        simlex_corr = results[('pmi', w, vc_file, 'simlex-999.txt')]
        print(f"pmi\t{w}\t{vc_file}\t{men_corr:.4f}\t{simlex_corr:.4f}")



print("1. Effect of window size:")
for vc_file in context_vocabs:
    men_trends = [results[('pmi', w, vc_file, 'men.txt')] for w in window_sizes]
    simlex_trends = [results[('pmi', w, vc_file, 'simlex-999.txt')] for w in window_sizes]
    print(f"  - With {vc_file}: MEN trend = {men_trends}, SimLex trend = {simlex_trends}")


Evaluating word vectors for different configurations...

Computing PMI word vectors (w=1, VC=vocab-15kws.txt)...
men.txt correlation: 0.4481
simlex-999.txt correlation: 0.2568

Computing PMI word vectors (w=1, VC=vocab-5k.txt)...
men.txt correlation: 0.3755
simlex-999.txt correlation: 0.2510

Computing PMI word vectors (w=3, VC=vocab-15kws.txt)...
men.txt correlation: 0.5340
simlex-999.txt correlation: 0.2108

Computing PMI word vectors (w=3, VC=vocab-5k.txt)...
men.txt correlation: 0.4626
simlex-999.txt correlation: 0.2159

Computing PMI word vectors (w=6, VC=vocab-15kws.txt)...
men.txt correlation: 0.5016
simlex-999.txt correlation: 0.1279

Computing PMI word vectors (w=6, VC=vocab-5k.txt)...
men.txt correlation: 0.4304
simlex-999.txt correlation: 0.1443

Summary of Results:
Method	Window	Context Vocab	MEN	SimLex-999
pmi	1	vocab-15kws.txt	0.4481	0.2568
pmi	1	vocab-5k.txt	0.3755	0.2510
pmi	3	vocab-15kws.txt	0.5340	0.2108
pmi	3	vocab-5k.txt	0.4626	0.2159
pmi	6	vocab-15kws.txt	0.5016	0.